This notebook builds the Gold layer by aggregating the cleansed Silver sales data into business level metrics suitable for reporting and analysis. Monthly aggregations are computed to derive total revenue, total units sold, and total number of orders, providing a concise and query optimized view of sales performance over time. The Gold dataset represents the final consumption layer of the data lake, designed for direct use by BI tools and analytical workloads.

In [1]:
# Import the necessary libraries 
from pyspark.sql import SparkSession
from pathlib import Path
import src.sqlqueries as sq
import sys
import os
import warnings
import utils.logger as logger
from pyspark.sql import functions as F
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql.functions import col, sum as spark_sum, countDistinct, avg, when

warnings.filterwarnings("ignore")


#Set the path for logging outputs
job_name = "gold_aggregations"
data_base_path = Path("../Logs") # path for logging data
data_working_path = os.path.join(data_base_path, job_name) 
os.makedirs(data_working_path, exist_ok=True)
logger.set_logging_path(data_working_path)

# Spark initialization locally for development
spark = (
    SparkSession.builder
    .appName("sales-gold-aggregations")
    .getOrCreate()
)

logger.log("Spark Session initialized")

df = spark.read.parquet(
    "../data/cleansed/sales"
)
logger.log("Spark DataFrame created from silver - cleansed sales parquet files")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/04 19:52:49 WARN Utils: Your hostname, gvidias, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/04 19:52:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/04 19:53:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-02-04 19:53:02: Spark Session initialized
2026-02-04 19:53:04: Spark DataFrame created from silver - cleansed sales parquet files


**Gold Aggregations**

In [2]:
logger.log("Creating monthly aggregations...")
monthly_gold = (
    df.groupBy("Order_Year", "Order_Month")
    .agg(
        spark_sum(col("Quantity_Ordered") * col("Price_Each")).alias("Total_Revenue"),
        spark_sum("Quantity_Ordered").alias("Total_Units_Sold"),
        countDistinct("Order_ID").alias("Total_Orders"),
        countDistinct("Product").alias("Unique_Products"),
        (spark_sum(col("Quantity_Ordered") * col("Price_Each")) / countDistinct("Order_ID")).alias("Avg_Order_Value")
    )
    .orderBy("Order_Year", "Order_Month")
)

2026-02-04 19:53:04: Creating monthly aggregations...


In [3]:
# Save to gold
monthly_gold.write.mode("overwrite").parquet("../data/gold/sales_monthly")
logger.log(f"Monthly aggregations saved: {monthly_gold.count()} rows")

# Preview
logger.log("Monthly aggregations preview:")
monthly_gold.show(12, truncate=False)

2026-02-04 19:53:08: Monthly aggregations saved: 13 rows
2026-02-04 19:53:08: Monthly aggregations preview:
+----------+-----------+------------------+----------------+------------+---------------+------------------+
|Order_Year|Order_Month|Total_Revenue     |Total_Units_Sold|Total_Orders|Unique_Products|Avg_Order_Value   |
+----------+-----------+------------------+----------------+------------+---------------+------------------+
|2019      |1          |1812742.8699999594|10852           |9262        |19             |195.7182973439818 |
|2019      |2          |2200012.2999999966|13425           |11496       |19             |191.37198155880276|
|2019      |3          |2804954.5699999304|16976           |14549       |19             |192.7936332393931 |
|2019      |4          |3389203.4699999234|20532           |17528       |19             |193.35939468278886|
|2019      |5          |3150537.6199999107|18641           |15836       |19             |198.947816367764  |
|2019      |6       

In [4]:
logger.log("Creating product aggregations...")
product_gold = (
    df.groupBy("Product")
    .agg(
        spark_sum(col("Quantity_Ordered") * col("Price_Each")).alias("Total_Revenue"),
        spark_sum("Quantity_Ordered").alias("Total_Units_Sold"),
        countDistinct("Order_ID").alias("Total_Orders"),
        avg("Price_Each").alias("Avg_Price")
    )
    .orderBy(col("Total_Revenue").desc())
)

# Save to gold
product_gold.write.mode("overwrite").parquet("../data/gold/sales_products")
logger.log(f"Product aggregations saved: {product_gold.count()} rows")

# Preview top 10
logger.log("Top 10 products by revenue:")
product_gold.show(10, truncate=False)

2026-02-04 19:53:09: Creating product aggregations...
2026-02-04 19:53:11: Product aggregations saved: 19 rows
2026-02-04 19:53:11: Top 10 products by revenue:
+--------------------------+------------------+----------------+------------+------------------+
|Product                   |Total_Revenue     |Total_Units_Sold|Total_Orders|Avg_Price         |
+--------------------------+------------------+----------------+------------+------------------+
|Macbook Pro Laptop        |8032500.0         |4725            |4721        |1700.0            |
|iPhone                    |4792900.0         |6847            |6840        |700.0             |
|ThinkPad Laptop           |4127958.719999969 |4128            |4126        |999.9899999999925 |
|Google Phone              |3317400.0         |5529            |5522        |600.0             |
|27in 4K Gaming Monitor    |2433147.6099999617|6239            |6225        |389.9899999999938 |
|34in Ultrawide Monitor    |2352898.079999963 |6192            |

In [6]:
logger.log("Creating quarterly aggregations...")
quarterly_gold = (
    df.withColumn(
        "Quarter",
        when(col("Order_Month").isin([1, 2, 3]), "Q1")
        .when(col("Order_Month").isin([4, 5, 6]), "Q2")
        .when(col("Order_Month").isin([7, 8, 9]), "Q3")
        .otherwise("Q4")
    )
    .groupBy("Order_Year", "Quarter")
    .agg(
        spark_sum(col("Quantity_Ordered") * col("Price_Each")).alias("Total_Revenue"),
        countDistinct("Order_ID").alias("Total_Orders"),
        spark_sum("Quantity_Ordered").alias("Total_Units_Sold")
    )
    .orderBy("Order_Year", "Quarter")
)

# Save to gold
quarterly_gold.write.mode("overwrite").parquet("../data/gold/sales_quarterly")
logger.log(f"Quarterly aggregations saved: {quarterly_gold.count()} rows")

# Preview
logger.log("Quarterly aggregations:")
quarterly_gold.show(truncate=False)

logger.log("GOLD LAYER COMPLETE - All aggregations created successfully")


2026-02-04 19:53:27: Creating quarterly aggregations...
2026-02-04 19:53:28: Quarterly aggregations saved: 5 rows
2026-02-04 19:53:28: Quarterly aggregations:
+----------+-------+--------------------+------------+----------------+
|Order_Year|Quarter|Total_Revenue       |Total_Orders|Total_Units_Sold|
+----------+-------+--------------------+------------+----------------+
|2019      |Q1     |6817709.740001427   |35307       |41253           |
|2019      |Q2     |9116006.300002813   |46353       |54405           |
|2019      |Q3     |6981930.9600016065  |36447       |42559           |
|2019      |Q4     |1.1540758160004579E7|60299       |70472           |
|2020      |Q1     |8670.289999999999   |31          |41              |
+----------+-------+--------------------+------------+----------------+

2026-02-04 19:53:29: GOLD LAYER COMPLETE - All aggregations created successfully
